# Tutorial 10 - CNNs & Vision Transformers

## Dr. David C. Schedl

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-Media/vco/blob/main/10_CNNs_ViTs.ipynb)

Note: this tutorial is geared towards students **experienced in programming** and aims to introduce you to **modern image-classification architectures in PyTorch**.

We train and compare four ideas on the same dataset (**CIFAR-10**):
1. **LeNet-5** - the classic convolutional network (1998).
2. **ConvNeXt** - a *modernised* CNN (2022).
3. **Vision Transformer (ViT)** - attention instead of convolution (2020).
4. **Transfer learning** - reuse a network pretrained on ImageNet.

Everything is sized to run on a **Colab GPU in a few minutes**. For training, go to **Edit -> Notebook settings -> Hardware accelerator -> GPU**.

## Setup

As a first step, we import the necessary libraries and pick a device. The only extra package we need is `torchinfo` (a `model.summary()` like TensorFlow has).

In [ ]:
!pip install -q torchinfo  # pretty model summaries

import os, urllib.request, shutil
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchinfo import summary
from tqdm.auto import tqdm

torch.manual_seed(0); np.random.seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    print("Using CPU! Training will be slow -- enable a GPU in Colab. :(")
else:
    print("Using GPU:", torch.cuda.get_device_name(0))

# ---- global training knob -------------------------------------------------
# Number of epochs every model is trained for. Raise this (e.g. 15-25) for
# better accuracy at the cost of runtime; 5 keeps the full run to a few minutes.
EPOCHS = 5

## 1. Data - CIFAR-10

CIFAR-10 holds 60,000 tiny 32x32 colour images in 10 classes. We load it via `torchvision`.

To keep the tutorial fast we train on a **subset** by default. Set `TRAIN_SUBSET = None` (and `TEST_SUBSET = None`) to use the full dataset for better accuracy.

In [ ]:
batch_size   = 128
TRAIN_SUBSET = None    # set to an int (e.g. 15000) to train on a subset for quicker runs
TEST_SUBSET  = None    # set to an int (e.g. 3000) to evaluate on a subset

# CIFAR-10's official host returns HTTP 403 on Colab, so we fetch the dataset from
# our course mirror. torchvision verifies the md5 and uses this local copy.
os.makedirs('./data', exist_ok=True)
cifar_archive = './data/cifar-10-python.tar.gz'
if not os.path.exists(cifar_archive):
    mirror_url = 'https://github.com/Digital-Media/cv_data/releases/download/cifar-10/cifar-10-python.tar.gz'
    req = urllib.request.Request(mirror_url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as r, open(cifar_archive, 'wb') as f:
        shutil.copyfileobj(r, f)

# light data augmentation for training, plain normalization for testing
norm = transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
tf_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                               transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(), norm])
tf_test  = transforms.Compose([transforms.ToTensor(), norm])

trainset = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=tf_train)
testset  = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=tf_test)

if TRAIN_SUBSET: trainset = torch.utils.data.Subset(trainset, range(TRAIN_SUBSET))
if TEST_SUBSET:  testset  = torch.utils.data.Subset(testset,  range(TEST_SUBSET))

train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True,  num_workers=0)
test_loader  = torch.utils.data.DataLoader(testset,  batch_size=batch_size, shuffle=False, num_workers=0)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
print(f"train: {len(trainset)} images | test: {len(testset)} images")

def imshow(img):
    img = img / 2 + 0.5  # unnormalize
    plt.imshow(np.transpose(img.numpy(), (1, 2, 0)))
    plt.axis('off')

imgs, labels = next(iter(train_loader))
plt.figure(figsize=(12, 2))
plt.title(' | '.join('%5s' % classes[labels[j]] for j in range(10)))
imshow(torchvision.utils.make_grid(imgs[:10], nrow=10)); plt.show()

## 2. Training utilities

A small reusable training loop (`train_model`), plus helpers to count parameters, plot the learning curves, and display predictions. We use the **AdamW** optimizer with a cosine learning-rate schedule and cross-entropy loss.

In [ ]:
def train_model(model, loaders, epochs=EPOCHS, lr=1e-3, weight_decay=1e-4, label="model"):
    train_loader, test_loader = loaders
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.CrossEntropyLoss()
    hist  = {"train_acc": [], "test_acc": [], "train_loss": [], "test_loss": []}

    for ep in range(1, epochs + 1):
        model.train(); rl = rc = rt = 0
        for x, y in tqdm(train_loader, desc=f"[{label}] ep {ep}/{epochs}", leave=False):
            x, y = x.to(device), y.to(device)
            logits = model(x); loss = crit(logits, y)
            opt.zero_grad(); loss.backward(); opt.step()
            rl += loss.item() * x.size(0)
            rc += (logits.argmax(1) == y).sum().item(); rt += x.size(0)
        sched.step()
        train_loss, train_acc = rl / rt, rc / rt

        model.eval(); tl = tc = tt = 0
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                tl += crit(logits, y).item() * x.size(0)
                tc += (logits.argmax(1) == y).sum().item(); tt += x.size(0)
        test_loss, test_acc = tl / tt, tc / tt

        hist["train_loss"].append(train_loss); hist["train_acc"].append(train_acc)
        hist["test_loss"].append(test_loss);   hist["test_acc"].append(test_acc)
        print(f"  ep {ep:2d}/{epochs}  train={train_acc:.3f}  test={test_acc:.3f}  loss={train_loss:.3f}/{test_loss:.3f}")
    return hist


def param_count(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


def plot_history(hist, label):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
    ep = range(1, len(hist["train_acc"]) + 1)
    ax[0].plot(ep, hist["train_loss"], label="train"); ax[0].plot(ep, hist["test_loss"], label="test")
    ax[0].set_title(f"{label} - loss");     ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(ep, hist["train_acc"], label="train"); ax[1].plot(ep, hist["test_acc"], label="test")
    ax[1].set_title(f"{label} - accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].set_ylim(0, 1)
    plt.tight_layout(); plt.show()


def show_predictions(model, n=20):
    """Display n test images with predicted [true] labels; green = correct, red = wrong."""
    model.eval()
    imgs, labels = next(iter(test_loader))
    with torch.no_grad():
        preds = model(imgs[:n].to(device)).argmax(1).cpu()
    fig = plt.figure(figsize=(14, 1.6 * ((n + 9) // 10)))
    for i in range(n):
        ax = fig.add_subplot((n + 9) // 10, 10, i + 1, xticks=[], yticks=[])
        ax.imshow(np.transpose((imgs[i] / 2 + 0.5).numpy(), (1, 2, 0)))
        ok = preds[i] == labels[i]
        ax.set_title(f"{classes[preds[i]]}\n[{classes[labels[i]]}]",
                     color='green' if ok else 'red', fontsize=8)
    plt.tight_layout(); plt.show()

## 3. LeNet-5 - the classic CNN (1998)

LeNet-5 (Yann LeCun et al.) is the blueprint every modern CNN descends from: alternating **convolutions** and **pooling** to extract features, followed by **fully-connected** layers to classify. Below is a lightly modernised variant (ReLU instead of sigmoid). Set `legacy=True` to get the original sigmoid version.

Inspired by [this blog post](https://towardsdatascience.com/implementing-yann-lecuns-lenet-5-in-pytorch-5e05a0911320).

In [ ]:
class LeNet(nn.Module):
    def __init__(self, input_shape=(3, 32, 32), nb_classes=10, legacy=False):
        super().__init__()
        self.act = nn.Sigmoid() if legacy else nn.ReLU()
        c1, c2 = (6, 16) if legacy else (20, 50)
        self.conv1 = nn.Conv2d(input_shape[0], c1, kernel_size=5, stride=1, padding=2)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(c1, c2, kernel_size=5, stride=1, padding=2)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_shape[1] // 4 * input_shape[2] // 4 * c2, 500)
        self.fc2 = nn.Linear(500, nb_classes)

    def forward(self, x):
        x = self.pool1(self.act(self.conv1(x)))
        x = self.pool2(self.act(self.conv2(x)))
        x = self.flatten(x)
        x = self.fc2(self.act(self.fc1(x)))
        return x

lenet = LeNet().to(device)
summary(lenet, (1, 3, 32, 32))

In [ ]:
hist_lenet = train_model(lenet, (train_loader, test_loader), label="LeNet")
plot_history(hist_lenet, "LeNet-5")
show_predictions(lenet)

## 4. ConvNeXt - a modernised CNN (2022)

[ConvNeXt](https://arxiv.org/abs/2201.03545) asks: *how far can a pure CNN go if we borrow the best tricks from Transformers?* The key ingredients of a ConvNeXt block are a **large-kernel depthwise convolution** (7x7), **LayerNorm**, an **inverted bottleneck** MLP, and **GELU** - all wrapped in a **residual** connection.

Two details matter a lot in practice and are easy to get wrong on small images:
- **LayerScale** - a learnable per-channel scale initialised near zero, so each block starts as a near-identity mapping. This stabilises training and noticeably improves accuracy.
- **Enough spatial resolution** - the original ConvNeXt uses a 4x4 "patchify" stem, but on tiny 32x32 CIFAR that (plus downsampling) shrinks the feature map to 2x2, where a 7x7 conv is almost all padding. We use a small **patch-2 stem** so the large-kernel convolutions actually have something to look at.

In [ ]:
IMG_SIZE, NUM_CLASSES = 32, 10

class ConvNeXtBlock(nn.Module):
    """DWConv 7x7 -> LN -> Linear(x4) -> GELU -> Linear(/4), scaled by LayerScale, + residual."""
    def __init__(self, dim, expansion=4, layer_scale_init=1e-6):
        super().__init__()
        self.dwconv  = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm    = nn.LayerNorm(dim)
        self.pwconv1 = nn.Linear(dim, dim * expansion)
        self.act     = nn.GELU()
        self.pwconv2 = nn.Linear(dim * expansion, dim)
        self.gamma   = nn.Parameter(layer_scale_init * torch.ones(dim))   # LayerScale

    def forward(self, x):
        r = x
        x = self.dwconv(x).permute(0, 2, 3, 1)      # (B,C,H,W) -> (B,H,W,C) for LN over C
        x = self.pwconv2(self.act(self.pwconv1(self.norm(x))))
        x = self.gamma * x                          # per-channel LayerScale
        return x.permute(0, 3, 1, 2) + r            # back to (B,C,H,W)


class Downsample(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.norm = nn.LayerNorm(in_dim)
        self.conv = nn.Conv2d(in_dim, out_dim, kernel_size=2, stride=2)
    def forward(self, x):
        x = self.norm(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)
        return self.conv(x)


class SimpleConvNeXt(nn.Module):
    def __init__(self, dims=(48, 96, 192), depths=(2, 2, 2), stem_patch=2):
        super().__init__()
        # small patch-2 stem keeps resolution on 32x32 images (32 -> 16 -> 8 -> 4)
        self.stem = nn.Conv2d(3, dims[0], kernel_size=stem_patch, stride=stem_patch)
        self.stages, self.downs = nn.ModuleList(), nn.ModuleList()
        for i, (dim, depth) in enumerate(zip(dims, depths)):
            self.stages.append(nn.Sequential(*[ConvNeXtBlock(dim) for _ in range(depth)]))
            if i < len(dims) - 1:
                self.downs.append(Downsample(dim, dims[i + 1]))
        self.head_norm = nn.LayerNorm(dims[-1])
        self.head      = nn.Linear(dims[-1], NUM_CLASSES)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        for i, stage in enumerate(self.stages):
            x = stage(x)
            if i < len(self.downs):
                x = self.downs[i](x)
        x = self.head_norm(x.mean([-2, -1]))        # global average pool
        return self.head(x)

convnext = SimpleConvNeXt().to(device)
print(f"SimpleConvNeXt - {param_count(convnext):,} params")
hist_cx = train_model(convnext, (train_loader, test_loader), label="ConvNeXt")
plot_history(hist_cx, "SimpleConvNeXt")

## 5. Vision Transformer (from scratch)

A [Vision Transformer](https://arxiv.org/abs/2010.11929) drops convolutions entirely. It cuts the image into **patches**, embeds each patch as a token (plus a learnable **[CLS]** token and **positional embeddings**), and processes the sequence with standard Transformer blocks of **multi-head self-attention** + MLP. The [CLS] token is read out for classification.

Because CIFAR images are tiny we use small **4x4 patches** (-> 64 tokens) and a compact model.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.h = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.qkv  = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x, return_attn=False):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]                 # (B, h, N, d_h)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(-1)
        out  = (attn @ v).transpose(1, 2).reshape(B, N, D)
        out  = self.proj(out)
        return (out, attn) if return_attn else out


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads=4, mlp_ratio=2.0):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attn = MultiHeadSelfAttention(dim, num_heads)
        self.mlp  = nn.Sequential(nn.Linear(dim, int(dim * mlp_ratio)), nn.GELU(),
                                  nn.Linear(int(dim * mlp_ratio), dim))
    def forward(self, x, return_attn=False):
        a, attn = self.attn(self.norm1(x), return_attn=True)
        x = x + a
        x = x + self.mlp(self.norm2(x))
        return (x, attn) if return_attn else x


class SimpleViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, dim=128, depth=4, num_heads=4, num_classes=10):
        super().__init__()
        self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size)
        n_patches = (img_size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.blocks = nn.ModuleList([TransformerBlock(dim, num_heads) for _ in range(depth)])
        self.norm   = nn.LayerNorm(dim)
        self.head   = nn.Linear(dim, num_classes)

    def forward(self, x, return_attn=False):
        B = x.size(0)
        x = self.patch_embed(x).flatten(2).transpose(1, 2)            # (B, N, dim)
        x = torch.cat([self.cls_token.expand(B, -1, -1), x], dim=1) + self.pos_embed
        attn_maps = []
        for blk in self.blocks:
            x, attn = blk(x, return_attn=True)
            attn_maps.append(attn)
        logits = self.head(self.norm(x)[:, 0])
        return (logits, attn_maps) if return_attn else logits

vit = SimpleViT().to(device)
print(f"SimpleViT - {param_count(vit):,} params")
hist_vit = train_model(vit, (train_loader, test_loader), lr=3e-4, label="ViT")
plot_history(hist_vit, "SimpleViT")

### What does the ViT attend to?

We can inspect the self-attention weights. **Attention rollout** (Abnar & Zuidema, 2020) multiplies the attention matrices across all layers (adding the residual connection as identity) to estimate how much the output [CLS] token draws from each input patch. Brighter = more attended.

In [ ]:
@torch.no_grad()
def attention_rollout(model, img):
    model.eval()
    _, attn_maps = model(img.unsqueeze(0).to(device), return_attn=True)
    result = torch.eye(attn_maps[0].size(-1), device=device)
    for attn in attn_maps:
        a = attn[0].mean(0)                      # average over heads -> (N+1, N+1)
        a = a + torch.eye(a.size(0), device=device)
        a = a / a.sum(-1, keepdim=True)
        result = a @ result
    mask = result[0, 1:]                         # CLS -> patches
    n = int(mask.numel() ** 0.5)
    return mask.reshape(n, n).cpu().numpy()

imgs, labels = next(iter(test_loader))
fig, axs = plt.subplots(2, 6, figsize=(13, 4.5))
for j in range(6):
    img = imgs[j]
    roll = attention_rollout(vit, img)
    roll = np.kron(roll, np.ones((4, 4)))        # upsample 8x8 -> 32x32
    axs[0, j].imshow(np.transpose((img / 2 + 0.5).numpy(), (1, 2, 0))); axs[0, j].axis('off')
    axs[0, j].set_title(classes[labels[j]], fontsize=9)
    axs[1, j].imshow(np.transpose((img / 2 + 0.5).numpy(), (1, 2, 0)))
    axs[1, j].imshow(roll, cmap='jet', alpha=0.5); axs[1, j].axis('off')
plt.suptitle("Attention rollout: where the [CLS] token looks"); plt.tight_layout(); plt.show()

## 6. Transfer Learning - reuse a pretrained network

Training from scratch needs lots of data. In practice we usually start from a network **pretrained on ImageNet** (1.2M images) and adapt it to our task. Here we take a **ConvNeXt-Tiny** - the grown-up sibling of the small ConvNeXt we built by hand in section 4 - freeze its backbone, and train only a fresh classification head, i.e. use it as a *fixed feature extractor*. This reaches good accuracy in just a couple of epochs.

> Quote from the [CS231n notes](https://cs231n.github.io/transfer-learning/): *"very few people train an entire ConvNet from scratch ... it is common to pretrain a ConvNet on a very large dataset ... and then use the ConvNet either as an initialization or a fixed feature extractor for the task of interest."*

Pretrained ImageNet models expect larger, ImageNet-normalised inputs, so we build a separate set of loaders that resize CIFAR-10 to 64x64.

In [ ]:
from torchvision import models

imagenet_norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
tf_tl = transforms.Compose([transforms.Resize(64), transforms.ToTensor(), imagenet_norm])

tl_train = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=tf_tl)
tl_test  = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=tf_tl)
tl_train = torch.utils.data.Subset(tl_train, range(5000))   # small subset -> fast
tl_test  = torch.utils.data.Subset(tl_test,  range(1000))
tl_loaders = (torch.utils.data.DataLoader(tl_train, batch_size=128, shuffle=True,  num_workers=0),
              torch.utils.data.DataLoader(tl_test,  batch_size=128, shuffle=False, num_workers=0))

convnext_tl = models.convnext_tiny(weights="IMAGENET1K_V1")
for p in convnext_tl.parameters():              # freeze the backbone
    p.requires_grad = False
in_features = convnext_tl.classifier[2].in_features
convnext_tl.classifier[2] = nn.Linear(in_features, 10)   # fresh head (trainable)
convnext_tl = convnext_tl.to(device)
print(f"ConvNeXt-Tiny - {param_count(convnext_tl):,} trainable params (head only)")

hist_tl = train_model(convnext_tl, tl_loaders, lr=1e-3, label="ConvNeXt-T-TL")
plot_history(hist_tl, "ConvNeXt-Tiny (transfer)")

## 7. Wrap-up - comparing the models

All four were trained on the same data budget. Note the trade-offs: LeNet is tiny but limited; ConvNeXt and the ViT are stronger but data-hungry from scratch; transfer learning gets the best accuracy almost for free by reusing ImageNet features. (Numbers depend on the subset size / epochs above - bump them up for a fairer fight.)

In [ ]:
rows = [("LeNet-5",        lenet,    hist_lenet),
        ("SimpleConvNeXt",  convnext, hist_cx),
        ("SimpleViT",       vit,      hist_vit),
        ("ConvNeXt-T (TL)", convnext_tl, hist_tl)]

print(f"{'model':<18}{'params':>12}{'best test acc':>16}")
print("-" * 46)
accs = []
for name, m, h in rows:
    best = max(h["test_acc"]); accs.append(best)
    print(f"{name:<18}{param_count(m):>12,}{best:>15.1%}")

plt.figure(figsize=(7, 3.5))
plt.bar([r[0] for r in rows], accs, color=['#4C72B0', '#55A868', '#C44E52', '#8172B2'])
plt.ylabel("best test accuracy"); plt.ylim(0, 1)
for i, a in enumerate(accs): plt.text(i, a + 0.02, f"{a:.0%}", ha='center')
plt.title("CIFAR-10 - architecture comparison"); plt.tight_layout(); plt.show()

## Try it yourself

1. **More data, more epochs.** Set `TRAIN_SUBSET = None` and increase the epochs. How does the gap between LeNet, ConvNeXt and the ViT change? The ViT in particular benefits a lot from more data.
2. **Patch size.** Change the ViT `patch_size` to 8 or 2. How do accuracy, token count and the attention maps change?
3. **Fine-tune vs. feature-extract.** In the transfer-learning section, unfreeze the ConvNeXt backbone (`p.requires_grad = True`) and train with a small learning rate (e.g. `1e-4`). Does full fine-tuning beat the frozen feature extractor?
4. **Your own block.** Add BatchNorm to LeNet, or a second ConvNeXt configuration, and compare.

In [ ]:
# Your code here